# TE02_001 - Robot Pose Tracking KPI

This notebook validates the KPI described in `detail.txt`: real-time tracking of the telehandler pose and relevant machine joints.

It records hardware joint data from `/joints` and `/encoders`, ROS model data from `/joint_states`, and the tracked telehandler pose from TF. The validation report checks joint accuracy, joint latency, pose update rate, and pose data gaps.

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import rclpy
import rclpy.time
from rclpy.duration import Duration
from rclpy.node import Node
from sensor_msgs.msg import JointState
from std_msgs.msg import Float32MultiArray
from tf2_ros import Buffer, TransformException, TransformListener

OUTPUT_DIR = os.path.join(
    os.environ.get('HOME', ''),
    'criarte_ws/scripts/Fortis_KPI/KPI/TE02_001',
)
print(f"{OUTPUT_DIR}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

JOINT_CSV = os.path.join(OUTPUT_DIR, 'kpi_tracking_joint_samples.csv')
POSE_CSV = os.path.join(OUTPUT_DIR, 'kpi_tracking_pose_samples.csv')
SUMMARY_CSV = os.path.join(OUTPUT_DIR, 'kpi_tracking_summary.csv')

BASE_FRAME = 'base_link'
TRACKED_FRAME = 'winch_link'
SAMPLE_PERIOD_S = 0.1  # 10 Hz KPI sampling

JOINT_LATENCY_LIMIT_MS = 200.0
JOINT_ACCURACY_LIMIT_PCT = 95.0
MIN_JOINT_MOTION_RANGE = 1e-3
VALIDATE_CAGE_AND_WINCH_JOINTS = True
POSE_MIN_RATE_HZ = 10.0
POSE_MAX_GAP_MS = 200.0
POSE_MAX_TF_AGE_P95_MS = 200.0
POSE_MAX_STALE_RATIO = 0.01
POSE_ENFORCE_STALE_RATIO = False
POSE_WARMUP_S = 2.0
DEG2RAD = math.pi / 180.0
FORK_COMPENSATION_K = 0.085
BOOM_RETRACTED_LENGTH_M = 6.391


In [2]:
ARM_JOINT_NAMES = [
    'base_to_cylinder_joint',
    'cylinder_to_arm1_joint',
    'arm_link1_to_arm2_joint',
    'arm_link2_to_arm3_joint',
    'arm_link3_to_arm4_joint',
    'arm_link4_to_armFork_joint',
]

CAGE_AND_WINCH_JOINT_NAMES = [
    'basket_link_to_armFork_joint',
    'basket_crane_link_to_basket_link',
    'winch_link_to_basket_crane_link',
]

VIRTUAL_JOINT_NAMES = [
    'basket_crane_link_to_basket_link',
]

IMPLEMENTED_CAGE_AND_WINCH_JOINT_NAMES = [
    name for name in CAGE_AND_WINCH_JOINT_NAMES
    if name not in VIRTUAL_JOINT_NAMES
]

JOINT_NAMES = ARM_JOINT_NAMES + CAGE_AND_WINCH_JOINT_NAMES
KPI_JOINT_NAMES = (
    ARM_JOINT_NAMES + IMPLEMENTED_CAGE_AND_WINCH_JOINT_NAMES
    if VALIDATE_CAGE_AND_WINCH_JOINTS
    else ARM_JOINT_NAMES
)


def msg_time_ns(node: Node, msg) -> int:
    stamp = getattr(getattr(msg, 'header', None), 'stamp', None)
    if stamp is not None and (stamp.sec or stamp.nanosec):
        return rclpy.time.Time.from_msg(stamp).nanoseconds
    return node.get_clock().now().nanoseconds


def compute_hardware_positions(latest_joints, latest_encoders):
    """Match telehandler_moveit/scripts/joint_remap.py real-machine mapping."""
    if latest_joints is None or len(latest_joints) < 4:
        return {name: np.nan for name in JOINT_NAMES}

    positions_data = latest_joints
    encoder_data = latest_encoders or []
    positions = [np.nan] * len(JOINT_NAMES)

    positions[0] = positions_data[2] * DEG2RAD
    positions[1] = -positions_data[0] * DEG2RAD

    extension_length = positions_data[1] - BOOM_RETRACTED_LENGTH_M
    prismatic_position = extension_length / 3.0
    positions[2] = -prismatic_position
    positions[3] = -prismatic_position
    positions[4] = -prismatic_position

    position_arm_link_to_end = positions_data[3] * DEG2RAD
    adjusted_position = position_arm_link_to_end + (positions[1] - FORK_COMPENSATION_K)
    positions[5] = -adjusted_position

    if len(positions_data) >= 5:
        positions[6] = positions_data[4] * DEG2RAD
    elif len(encoder_data) >= 1:
        positions[6] = encoder_data[0] * DEG2RAD

    if len(positions_data) >= 6:
        positions[7] = positions_data[5] * DEG2RAD

    if len(positions_data) >= 7:
        positions[8] = positions_data[6]
    elif len(encoder_data) >= 2:
        positions[8] = -encoder_data[1]

    return dict(zip(JOINT_NAMES, positions))


In [3]:
class KPITrackingLogger(Node):
    def __init__(self):
        super().__init__('te02_001_kpi_tracking_logger')

        self.latest_joints = None
        self.latest_encoders = None
        self.latest_ros_positions = {}
        self.joint_rows = []
        self.pose_rows = []
        self.joints_msg_count = 0
        self.encoders_msg_count = 0
        self.joint_states_msg_count = 0
        self.tf_success_count = 0
        self.tf_failure_count = 0
        self.sample_count = 0

        self.tf_buffer = Buffer(cache_time=Duration(seconds=10.0))
        self.tf_listener = TransformListener(self.tf_buffer, self)

        self.create_subscription(Float32MultiArray, '/joints', self.joints_cb, 10)
        self.create_subscription(Float32MultiArray, '/encoders', self.encoders_cb, 10)
        self.create_subscription(JointState, '/joint_states', self.joint_state_cb, 10)
        self.create_timer(SAMPLE_PERIOD_S, self.sample_cb)
        self.create_timer(2.0, self.health_cb)

        self.get_logger().info(
            f'Recording TE02_001 KPI data: joints plus TF {BASE_FRAME} -> {TRACKED_FRAME}'
        )

    def joints_cb(self, msg):
        self.latest_joints = list(msg.data)
        self.joints_msg_count += 1

    def encoders_cb(self, msg):
        self.latest_encoders = list(msg.data)
        self.encoders_msg_count += 1

    def joint_state_cb(self, msg):
        self.latest_ros_positions = dict(zip(msg.name, msg.position))
        self.joint_states_msg_count += 1

    def health_cb(self):
        missing = []
        if self.joints_msg_count == 0:
            missing.append('/joints')
        if self.joint_states_msg_count == 0:
            missing.append('/joint_states')
        if self.tf_success_count == 0:
            missing.append(f'TF {BASE_FRAME}->{TRACKED_FRAME}')

        if missing:
            self.get_logger().warn(
                'No data yet from: ' + ', '.join(missing) +
                '. Check joint_remap, robot_state_publisher, and TF.'
            )
        else:
            self.get_logger().info(
                f'Acquisition OK: /joints={self.joints_msg_count}, '
                f'/joint_states={self.joint_states_msg_count}, '
                f'tf_ok={self.tf_success_count}, tf_fail={self.tf_failure_count}'
            )

    def sample_cb(self):
        now_ns = self.get_clock().now().nanoseconds
        self.sample_count += 1

        hardware_positions = compute_hardware_positions(
            self.latest_joints, self.latest_encoders
        )

        for joint_name in KPI_JOINT_NAMES:
            hardware_value = hardware_positions[joint_name]
            ros_value = self.latest_ros_positions.get(joint_name, np.nan)
            self.joint_rows.append({
                'time_ns': now_ns,
                'joint_name': joint_name,
                'hardware_position': hardware_value,
                'ros_position': ros_value,
                'error': ros_value - hardware_value
                if np.isfinite(hardware_value) and np.isfinite(ros_value)
                else np.nan,
            })

        try:
            tf = self.tf_buffer.lookup_transform(
                BASE_FRAME, TRACKED_FRAME, rclpy.time.Time()
            )
            t = tf.transform.translation
            q = tf.transform.rotation
            tf_time_ns = rclpy.time.Time.from_msg(tf.header.stamp).nanoseconds
            self.tf_success_count += 1
            self.pose_rows.append({
                'sample_time_ns': now_ns,
                'tf_time_ns': tf_time_ns,
                'tf_age_ms': (now_ns - tf_time_ns) / 1e6 if tf_time_ns else np.nan,
                'frame_id': BASE_FRAME,
                'child_frame_id': TRACKED_FRAME,
                'x': t.x,
                'y': t.y,
                'z': t.z,
                'qx': q.x,
                'qy': q.y,
                'qz': q.z,
                'qw': q.w,
            })
        except TransformException as exc:
            self.tf_failure_count += 1
            self.pose_rows.append({
                'sample_time_ns': now_ns,
                'tf_time_ns': np.nan,
                'tf_age_ms': np.nan,
                'frame_id': BASE_FRAME,
                'child_frame_id': TRACKED_FRAME,
                'x': np.nan,
                'y': np.nan,
                'z': np.nan,
                'qx': np.nan,
                'qy': np.nan,
                'qz': np.nan,
                'qw': np.nan,
            })

    def save_to_csv(self):
        pd.DataFrame(self.joint_rows).to_csv(JOINT_CSV, index=False)
        pd.DataFrame(self.pose_rows).to_csv(POSE_CSV, index=False)
        print(
            f'Saved {len(self.joint_rows)} joint samples and {len(self.pose_rows)} pose samples. '
            f'/joints={self.joints_msg_count}, /encoders={self.encoders_msg_count}, '
            f'/joint_states={self.joint_states_msg_count}, '
            f'tf_ok={self.tf_success_count}, tf_fail={self.tf_failure_count}'
        )
        if self.joints_msg_count == 0 or self.joint_states_msg_count == 0:
            print('WARNING: joint CSV was saved without required topic data. Start joint_remap and verify /joints plus /joint_states before recording.')
        if self.tf_success_count == 0:
            print(f'WARNING: pose CSV was saved without TF data for {BASE_FRAME} -> {TRACKED_FRAME}.')

In [4]:
# Run this cell while the telehandler is moving. Stop/interupt it when the maneuver is complete.
if not rclpy.ok():
    rclpy.init()

kpi_logger = KPITrackingLogger()

try:
    print('Recording TE02_001 KPI data. Move the telehandler, then interrupt this cell to save CSV files.')
    rclpy.spin(kpi_logger)
except KeyboardInterrupt:
    print('Recording stopped. Saving KPI data...')
finally:
    kpi_logger.save_to_csv()
    kpi_logger.destroy_node()

Recording TE02_001 KPI data. Move the telehandler, then interrupt this cell to save CSV files.


[INFO] [1783426740.066772849] [te02_001_kpi_tracking_logger]: Recording TE02_001 KPI data: joints plus TF base_link -> winch_link
[INFO] [1783426742.038034683] [te02_001_kpi_tracking_logger]: Acquisition OK: /joints=18, /joint_states=17, tf_ok=18, tf_fail=2
[INFO] [1783426744.037769719] [te02_001_kpi_tracking_logger]: Acquisition OK: /joints=38, /joint_states=37, tf_ok=38, tf_fail=2
[INFO] [1783426746.037603543] [te02_001_kpi_tracking_logger]: Acquisition OK: /joints=58, /joint_states=57, tf_ok=58, tf_fail=2
[INFO] [1783426748.037737353] [te02_001_kpi_tracking_logger]: Acquisition OK: /joints=78, /joint_states=77, tf_ok=78, tf_fail=2
[INFO] [1783426750.038135372] [te02_001_kpi_tracking_logger]: Acquisition OK: /joints=98, /joint_states=97, tf_ok=98, tf_fail=2
[INFO] [1783426752.037657571] [te02_001_kpi_tracking_logger]: Acquisition OK: /joints=118, /joint_states=117, tf_ok=118, tf_fail=2
[INFO] [1783426754.037975391] [te02_001_kpi_tracking_logger]: Acquisition OK: /joints=138, /joint_s

Recording stopped. Saving KPI data...
Saved 13248 joint samples and 1472 pose samples. /joints=1469, /encoders=24200, /joint_states=1468, tf_ok=1470, tf_fail=2


In [5]:
def estimate_latency_ms(reference_values, tracked_values, sample_period_s):
    ref = np.asarray(reference_values, dtype=float)
    trk = np.asarray(tracked_values, dtype=float)
    valid = np.isfinite(ref) & np.isfinite(trk)
    ref = ref[valid]
    trk = trk[valid]

    if len(ref) < 5 or np.nanstd(ref) == 0 or np.nanstd(trk) == 0:
        return np.nan

    corr = np.correlate(trk - np.mean(trk), ref - np.mean(ref), mode='full')
    lags = np.arange(-len(ref) + 1, len(trk))
    lag_samples = lags[int(np.argmax(corr))]
    return abs(lag_samples * sample_period_s * 1000.0)


def validate_joint_tracking(df_joint):
    rows = []

    for joint_name, group in df_joint.groupby('joint_name'):
        group = group.sort_values('time_ns')
        valid = group.dropna(subset=['hardware_position', 'ros_position'])

        if joint_name not in KPI_JOINT_NAMES:
            reason = (
                'Virtual joint not implemented on the real machine'
                if joint_name in VIRTUAL_JOINT_NAMES
                else 'Excluded from this run; set VALIDATE_CAGE_AND_WINCH_JOINTS=True to include'
            )
            rows.append({
                'metric': f'joint:{joint_name}',
                'latency_ms': np.nan,
                'accuracy_pct': np.nan,
                'mae': np.nan,
                'samples': 0,
                'status': 'SKIP',
                'reason': reason,
            })
            continue

        if valid.empty:
            rows.append({
                'metric': f'joint:{joint_name}',
                'latency_ms': np.nan,
                'accuracy_pct': np.nan,
                'mae': np.nan,
                'samples': 0,
                'status': 'FAIL',
                'reason': 'No aligned hardware and ROS samples',
            })
            continue

        mae = float(np.mean(np.abs(valid['error'])))
        motion_range = float(valid['hardware_position'].max() - valid['hardware_position'].min())
        if motion_range < MIN_JOINT_MOTION_RANGE:
            rows.append({
                'metric': f'joint:{joint_name}',
                'latency_ms': np.nan,
                'accuracy_pct': np.nan,
                'mae': mae,
                'samples': len(valid),
                'status': 'SKIP',
                'reason': f'Not enough commanded/measured motion: range={motion_range:.6g}',
            })
            continue

        accuracy_pct = 100.0 - ((mae / motion_range) * 100.0)
        latency_ms = estimate_latency_ms(
            valid['hardware_position'], valid['ros_position'], SAMPLE_PERIOD_S
        )

        passes_accuracy = accuracy_pct >= JOINT_ACCURACY_LIMIT_PCT
        passes_latency = np.isfinite(latency_ms) and latency_ms <= JOINT_LATENCY_LIMIT_MS
        rows.append({
            'metric': f'joint:{joint_name}',
            'latency_ms': latency_ms,
            'accuracy_pct': accuracy_pct,
            'mae': mae,
            'samples': len(valid),
            'status': 'PASS' if passes_accuracy and passes_latency else 'FAIL',
            'reason': '',
        })

    return rows


def validate_pose_tracking(df_pose):
    valid_pose = df_pose.dropna(subset=['x', 'y', 'z']).copy()
    if valid_pose.empty:
        return [{
            'metric': 'pose:tf_tracking',
            'latency_ms': np.nan,
            'accuracy_pct': np.nan,
            'mae': np.nan,
            'samples': 0,
            'status': 'FAIL',
            'reason': f'No TF pose available for {BASE_FRAME} -> {TRACKED_FRAME}',
        }]

    start_ns = valid_pose['sample_time_ns'].min()
    valid_pose = valid_pose[
        (valid_pose['sample_time_ns'] - start_ns) >= POSE_WARMUP_S * 1e9
    ]
    if valid_pose.empty:
        return [{
            'metric': 'pose:tf_tracking',
            'latency_ms': np.nan,
            'accuracy_pct': np.nan,
            'mae': np.nan,
            'samples': 0,
            'status': 'FAIL',
            'reason': f'No valid TF pose after {POSE_WARMUP_S:.1f}s warmup',
        }]

    duration_s = (
        valid_pose['sample_time_ns'].max() - valid_pose['sample_time_ns'].min()
    ) / 1e9
    rate_hz = len(valid_pose) / duration_s if duration_s > 0 else 0.0
    gaps_ms = valid_pose['sample_time_ns'].diff().dropna() / 1e6
    max_gap_ms = float(gaps_ms.max()) if not gaps_ms.empty else 0.0
    tf_age_p95_ms = float(valid_pose['tf_age_ms'].quantile(0.95))
    stale_ratio = float((valid_pose['tf_age_ms'] > POSE_MAX_GAP_MS).mean())

    passes = (
        rate_hz >= POSE_MIN_RATE_HZ
        and max_gap_ms <= POSE_MAX_GAP_MS
        and tf_age_p95_ms <= POSE_MAX_TF_AGE_P95_MS
        and (
            stale_ratio <= POSE_MAX_STALE_RATIO
            if POSE_ENFORCE_STALE_RATIO
            else True
        )
    )

    return [{
        'metric': 'pose:tf_tracking',
        'latency_ms': tf_age_p95_ms,
        'accuracy_pct': np.nan,
        'mae': np.nan,
        'samples': len(valid_pose),
        'status': 'PASS' if passes else 'FAIL',
        'reason': f'rate_hz={rate_hz:.2f}, max_gap_ms={max_gap_ms:.2f}, stale_ratio={stale_ratio:.3f}',
    }]

In [9]:
def validate_kpi():
    if not os.path.exists(JOINT_CSV) or not os.path.exists(POSE_CSV):
        raise FileNotFoundError('Run the recording cell first to create KPI CSV files.')

    df_joint = pd.read_csv(JOINT_CSV)
    df_pose = pd.read_csv(POSE_CSV)

    summary_rows = validate_joint_tracking(df_joint) + validate_pose_tracking(df_pose)
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(SUMMARY_CSV, index=False)

    print('=== TE02_001 KPI VALIDATION REPORT ===')
    print(f'Joint latency KPI : <= {JOINT_LATENCY_LIMIT_MS:.0f} ms')
    print(f'Joint accuracy KPI: >= {JOINT_ACCURACY_LIMIT_PCT:.0f} %')
    print(f'Pose tracking KPI : >= {POSE_MIN_RATE_HZ:.0f} Hz, <= {POSE_MAX_GAP_MS:.0f} ms max gap, <= {POSE_MAX_TF_AGE_P95_MS:.0f} ms p95 TF age')
    print()

    for row in summary_rows:
        print(
            f"{row['metric']:<45} {row['status']:<4} "
            f"samples={row['samples']} "
            f"latency_ms={row['latency_ms']} "
            f"accuracy_pct={row['accuracy_pct']} "
            f"mae={row['mae']} "
            f"{row['reason']}"
        )

    scored = summary[summary['status'] != 'SKIP']
    overall = 'PASS' if not scored.empty and (scored['status'] == 'PASS').all() else 'FAIL'
    print(f'\nOverall TE02_001 result: {overall}')
    print(f'Summary written to: {SUMMARY_CSV}')
    return summary


summary = validate_kpi()
summary

=== TE02_001 KPI VALIDATION REPORT ===
Joint latency KPI : <= 200 ms
Joint accuracy KPI: >= 95 %
Pose tracking KPI : >= 10 Hz, <= 200 ms max gap, <= 200 ms p95 TF age

joint:arm_link1_to_arm2_joint                 PASS samples=1469 latency_ms=0.0 accuracy_pct=99.9921766203572 mae=0.00020490938069356483 
joint:arm_link2_to_arm3_joint                 PASS samples=1469 latency_ms=0.0 accuracy_pct=99.9921766203572 mae=0.00020490938069356483 
joint:arm_link3_to_arm4_joint                 PASS samples=1469 latency_ms=0.0 accuracy_pct=99.9921766203572 mae=0.00020490938069356483 
joint:arm_link4_to_armFork_joint              PASS samples=1469 latency_ms=0.0 accuracy_pct=99.98787734569335 mae=6.178142360499578e-05 
joint:base_to_cylinder_joint                  PASS samples=1469 latency_ms=0.0 accuracy_pct=99.98548499990483 mae=0.00018772088736021918 
joint:basket_crane_link_to_basket_link        FAIL samples=0 latency_ms=nan accuracy_pct=nan mae=nan No aligned hardware and ROS samples
joint:bas

,metric,latency_ms,accuracy_pct,mae,samples,status,reason
0,joint:arm_link1_to_arm2_joint,0.00000,99.992177,0.000205,1469,PASS,
1,joint:arm_link2_to_arm3_joint,0.00000,99.992177,0.000205,1469,PASS,
2,joint:arm_link3_to_arm4_joint,0.00000,99.992177,0.000205,1469,PASS,
3,joint:arm_link4_to_armFork_joint,0.00000,99.987877,0.000062,1469,PASS,
4,joint:base_to_cylinder_joint,0.00000,99.985485,0.000188,1469,PASS,
5,joint:basket_crane_link_to_basket_link,NaN,NaN,NaN,0,FAIL,No aligned hardware and ROS samples
6,joint:basket_link_to_armFork_joint,0.00000,99.954594,0.000454,1469,PASS,
7,joint:cylinder_to_arm1_joint,0.00000,99.992591,0.000038,1469,PASS,
8,joint:winch_link_to_basket_crane_link,0.00000,99.903514,0.005735,1469,PASS,
9,pose:tf_tracking,104.62679,NaN,NaN,1449,PASS,"rate_hz=10.01, max_gap_ms=101.21, stale_ratio=..."
